# Training

This is where the surrogate actually learns. Two terms make up the loss:
`MSE(pressure)` (the main pressure-field supervision, ~3600 targets per shape) and
`MSE(cd_pred, true_cd)` (the direct Cd supervision, but only 1 target per shape).

A third, "consistency" term was tried here and removed — see the note at the bottom
of this notebook for what it was, why it seemed promising, and why it actually made
both pressure and Cd worse when trained jointly with everything else. It's revisited
properly, as a separate fine-tuning stage, in `finetune_cd_head.ipynb`.


Load the cached datasets from the previous notebook. The batch size (4) isn't an
arbitrary choice — 8 was tried first, but with 4 attention heads per `FeaStConv`
layer it pushed VRAM usage past what the development GPU (6 GB) had available, and
training silently fell back to slow shared system memory: about 15x slower per epoch,
with no error or warning. Worth checking for on any GPU-memory-constrained machine.


In [1]:
import sys, json
sys.path.insert(0, "..")

import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from src.model import MeshSurrogate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device)

train_data = torch.load("../outputs/cache/train_pyg.pt", weights_only=False)
val_data = torch.load("../outputs/cache/val_pyg.pt", weights_only=False)
test_data = torch.load("../outputs/cache/test_pyg.pt", weights_only=False)
with open("../outputs/norm_stats.json") as f:
    stats = json.load(f)

train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
val_loader = DataLoader(val_data, batch_size=4)
test_loader = DataLoader(test_data, batch_size=4)


device cuda


The model, optimizer, and one epoch of training/evaluation as a function. Losses are
tracked in normalized units (for the optimizer and scheduler) but reported back in
original units, since "MSE of a standardized value" isn't something you can
sanity-check by eye.


In [2]:
model = MeshSurrogate(in_channels=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

LAMBDA_CD = 1.0
pressure_std = stats["pressure_std"]
cd_std = stats["cd_std"]


def run_epoch(loader, train: bool):
    model.train(train)
    total_p_loss, total_cd_loss, n = 0.0, 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        if train:
            optimizer.zero_grad()
        pressure_pred, cd_pred = model(batch.x, batch.edge_index, batch.batch)
        p_loss = F.mse_loss(pressure_pred, batch.y_pressure)
        cd_loss = F.mse_loss(cd_pred, batch.y_cd.squeeze(-1))
        loss = p_loss + LAMBDA_CD * cd_loss
        if train:
            loss.backward()
            optimizer.step()
        bs = batch.num_graphs
        total_p_loss += p_loss.item() * bs
        total_cd_loss += cd_loss.item() * bs
        n += bs
    return (total_p_loss / n) * (pressure_std ** 2), (total_cd_loss / n) * (cd_std ** 2)


The training loop itself: 200 epochs, saving the checkpoint whenever validation loss
improves, and also writing progress to a plain text log file — useful when running
this non-interactively, since a notebook's own cell output only gets written back to
disk once the whole cell finishes.


In [3]:
import os, time
os.makedirs("../outputs/checkpoints", exist_ok=True)

history = {"train_pressure": [], "val_pressure": [], "train_cd": [], "val_cd": []}
best_val = float("inf")
N_EPOCHS = 200

log_path = "../outputs/training_log.txt"
with open(log_path, "w") as f:
    f.write("epoch,train_p,val_p,train_cd,val_cd,elapsed_sec\n")

t_start = time.time()
for epoch in range(N_EPOCHS):
    train_p, train_cd = run_epoch(train_loader, train=True)
    with torch.no_grad():
        val_p, val_cd = run_epoch(val_loader, train=False)

    val_loss = val_p / (pressure_std ** 2) + val_cd / (cd_std ** 2)
    scheduler.step(val_loss)

    history["train_pressure"].append(train_p)
    history["val_pressure"].append(val_p)
    history["train_cd"].append(train_cd)
    history["val_cd"].append(val_cd)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "../outputs/checkpoints/best_model.pt")

    with open(log_path, "a") as f:
        f.write(f"{epoch},{train_p:.4f},{val_p:.4f},{train_cd:.4f},{val_cd:.4f},{time.time()-t_start:.1f}\n")

    if epoch % 10 == 0 or epoch == N_EPOCHS - 1:
        print(f"epoch {epoch:3d}  train_p {train_p:.4f}  val_p {val_p:.4f}  train_cd {train_cd:.4f}  val_cd {val_cd:.4f}")

with open("../outputs/history.json", "w") as f:
    json.dump(history, f)

with open(log_path, "a") as f:
    f.write("DONE\n")


epoch   0  train_p 317.6439  val_p 265.6006  train_cd 59.5944  val_cd 56.5527


epoch  10  train_p 108.9461  val_p 135.4238  train_cd 12.8900  val_cd 16.6139


epoch  20  train_p 88.0345  val_p 116.8927  train_cd 5.5161  val_cd 29.3929


epoch  30  train_p 76.9258  val_p 97.2529  train_cd 2.2845  val_cd 4.9779


epoch  40  train_p 72.6342  val_p 94.4246  train_cd 1.3990  val_cd 5.0155


...

epoch 160  train_p 53.2133  val_p 74.8208  train_cd 0.0612  val_cd 4.4016


epoch 170  train_p 53.0707  val_p 74.4530  train_cd 0.0641  val_cd 4.3953


epoch 180  train_p 53.2290  val_p 75.6575  train_cd 0.0591  val_cd 4.3696


epoch 190  train_p 53.2256  val_p 74.9341  train_cd 0.0750  val_cd 4.4342


epoch 199  train_p 52.7283  val_p 74.9851  train_cd 0.0612  val_cd 4.3917


Finally, load the best checkpoint (not necessarily the last epoch) and check it
against the held-out test set — the number that actually matters, since it's the only
one the model never influenced during training.


In [4]:
model.load_state_dict(torch.load("../outputs/checkpoints/best_model.pt", weights_only=True))
with torch.no_grad():
    test_p, test_cd = run_epoch(test_loader, train=False)
print(f"test pressure MSE (orig units) {test_p:.4f}, test cd MSE (orig units) {test_cd:.4f}")


test pressure MSE (orig units) 72.5394, test cd MSE (orig units) 5.4273


## A negative result worth keeping: the consistency-loss attempt

A natural idea for narrowing the Cd head's disagreement with the pressure head (see
the README) is to add a third training term: recompute Cd from the model's own
predicted pressure using the same analytic formula that built the training labels,
and penalize the Cd head for disagreeing with it — on every training batch, not
just the ones with a real Cd label. It was implemented (`differentiable_drag_proxy_batch`
in `src/shape_opt.py`), tuned (Huber loss instead of MSE, weight warmup over the
first 20 epochs) to fix instability found in an initial dry run, then trained fully.

**It made both metrics worse**: test pressure MSE went from 68.2 to 79.7, test Cd MSE
from 4.5 to 8.0 — while only narrowing the shape-optimization head-disagreement gap
by about 6% relative. Two causes, diagnosed from the training curves: the "teacher"
signal (Cd derived from predicted pressure) never became fully reliable during joint
training, since it depends on a pressure head that itself only converges gradually;
and although the *target* was detached, `cd_pred` still shares the encoder with the
pressure head, so this noisy auxiliary gradient flowed into the encoder and degraded
pressure predictions too — not just Cd.

The fix is a separate fine-tuning stage instead of a joint training term: train this
model normally (as above), then freeze the encoder and pressure head entirely and
fine-tune only the small Cd-head MLP against the same consistency signal — now a
reliable teacher from step one, with no shared-encoder path left for it to disrupt
anything else through. See `finetune_cd_head.ipynb`.
